[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/29_cnn/90_exercises.ipynb)

# Bài tập — Chương 29: CNN (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/29_cnn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_shapes

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
rng = np.random.default_rng(0)
to_tensor = lambda a: torch.tensor(a[:, None], dtype=torch.float32)

## L0 — Nhớ khái niệm

**Câu 1.** (a) Công thức kích thước đầu ra của conv và số tham số của `Conv2d(C_in, C_out, k)`. (b) Vì sao CNN cần ít tham số hơn MLP với ảnh? (c) Phân biệt đồng biến và bất biến với phép dịch; thành phần nào của CNN tạo ra mỗi tính chất? (d) Tác dụng của conv 1×1. (e) Khối residual là gì và giải quyết vấn đề nào?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một nhóm phân loại ảnh **chữ số viết tay** dùng augmentation: lật ngang, lật dọc, xoay ngẫu nhiên 0–360°. Kết quả kém hơn không augmentation, đặc biệt nhầm 6↔9 và 2↔5. Vì sao? Đề xuất augmentation phù hợp; và cho bài toán phân loại ảnh vệ tinh (nhìn từ trên xuống) thì sao?

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Đầu vào 3×64×64 → `Conv2d(3, 16, 5, stride=1, padding=2)` → `MaxPool2d(2)` → `Conv2d(16, 32, 3, stride=2, padding=1)` → `AdaptiveAvgPool2d(1)` → `Linear(32, 10)`. Tính shape sau mỗi lớp (bỏ batch) và tổng số tham số.

In [ ]:
### START CODE HERE ### (≈ 6 dòng)
shape_conv1 = None
shape_pool = None
shape_conv2 = None
shape_gap = None
total_params3 = None
### END CODE HERE ###
print(shape_conv1, shape_pool, shape_conv2, shape_gap, total_params3)

In [ ]:
net3 = nn.Sequential(nn.Conv2d(3, 16, 5, padding=2), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, stride=2, padding=1),
                     nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10))
x = torch.zeros(1, 3, 64, 64)
shapes = []
for layer in net3:
    x = layer(x)
    shapes.append(tuple(x.shape[1:]))
check_equal(shape_conv1, shapes[0], "sau conv1")
check_equal(shape_pool, shapes[1], "sau pool")
check_equal(shape_conv2, shapes[2], "sau conv2")
check_equal(shape_gap, shapes[3], "sau GAP")
check_equal(total_params3, sum(p.numel() for p in net3.parameters()), "tổng tham số")

**Câu 4.** Ảnh 4×4 và kernel 2×2, stride 2, không padding:
$$X = \begin{bmatrix}1 & 2 & 0 & 1\\3 & 1 & 1 & 0\\0 & 2 & 4 & 1\\1 & 0 & 2 & 3\end{bmatrix}, \quad K = \begin{bmatrix}1 & 0\\0 & -1\end{bmatrix}$$
Tính đầu ra tích chập (cross-correlation) 2×2 và max pooling 2×2 của $X$.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
conv4 = None
pool4 = None
### END CODE HERE ###
print(conv4, pool4, sep="\n")

In [ ]:
X4 = torch.tensor([[1, 2, 0, 1], [3, 1, 1, 0], [0, 2, 4, 1], [1, 0, 2, 3]], dtype=torch.float32)[None, None]
K4 = torch.tensor([[1, 0], [0, -1]], dtype=torch.float32)[None, None]
check_close(conv4, F.conv2d(X4, K4, stride=2)[0, 0].numpy(), message="tích chập stride 2")
check_close(pool4, F.max_pool2d(X4, 2)[0, 0].numpy(), message="max pooling")

## L3 — Code

**Bài 5.** Viết `conv2d_forward(x, w, b, stride, padding)` cho **batch**: `x` (N, C_in, H, W), `w` (C_out, C_in, k, k), `b` (C_out,) → (N, C_out, H_out, W_out), vector hoá (im2col: dùng slicing theo $k^2$ vị trí trong kernel, không lặp theo pixel). So với `F.conv2d`.

In [ ]:
def conv2d_forward(x, w, b, stride=1, padding=0):
    ### START CODE HERE ### (≈ 12 dòng)
    # pad x; H_out, W_out theo công thức; cols shape (N, C_in, k, k, H_out, W_out) điền bằng slicing với bước stride
    # out = einsum("nckl hw, ockl -> nohw") + b
    pass
    ### END CODE HERE ###

In [ ]:
x5 = rng.normal(size=(4, 3, 13, 11))
w5, b5 = rng.normal(size=(6, 3, 3, 3)), rng.normal(size=6)
for stride, padding in [(1, 0), (1, 1), (2, 1), (3, 2)]:
    ours = conv2d_forward(x5, w5, b5, stride, padding)
    ref = F.conv2d(torch.tensor(x5), torch.tensor(w5), torch.tensor(b5), stride=stride, padding=padding).numpy()
    check_close(ours, ref, tol=1e-10, message=f"stride {stride}, padding {padding}: khớp F.conv2d {ref.shape}")

**Bài 6.** Viết `maxpool_forward(x, size)` (stride = size, x: (N, C, H, W), H, W chia hết cho size) trả về `(out, mask)` và `maxpool_backward(d_out, mask, size)`: gradient chỉ chảy về **vị trí lớn nhất** trong mỗi ô. Kiểm tra với autograd.

In [ ]:
def maxpool_forward(x, size=2):
    ### START CODE HERE ### (≈ 5 dòng)
    return None, None   # gợi ý: reshape (N, C, H/s, s, W/s, s); max theo trục 3 và 5; mask = (x_reshaped == max)
    ### END CODE HERE ###


def maxpool_backward(d_out, mask, size=2):
    ### START CODE HERE ### (≈ 3 dòng)
    return None   # phân phối d_out về các vị trí mask = True; reshape về (N, C, H, W)
    ### END CODE HERE ###

In [ ]:
x6 = rng.normal(size=(2, 3, 8, 6))                                       # dữ liệu liên tục → không có giá trị bằng nhau
out6, mask6 = maxpool_forward(x6)
d6 = rng.normal(size=out6.shape)
dx6 = maxpool_backward(d6, mask6)
tx6 = torch.tensor(x6, requires_grad=True)
tout = F.max_pool2d(tx6, 2)
tout.backward(torch.tensor(d6))
check_close(out6, tout.detach().numpy(), message="forward")
check_close(dx6, tx6.grad.numpy(), message="backward: gradient về đúng vị trí max")
check_equal(int((dx6 != 0).sum()), int(out6.size), "mỗi ô 2×2 chỉ một phần tử nhận gradient")

## L4 — Debug

**Bài 7.** CNN dưới đây cho bộ shapes có lỗi: (1) dữ liệu thiếu **chiều kênh**; (2) không padding nên kích thước giảm khác dự kiến, nhưng `Linear` vẫn khai báo `32 * 7 * 7` → lỗi shape; (3) đánh giá ở chế độ train (có Dropout) và tính accuracy trên **tập train**. Sửa (giữ nguyên các lớp conv không padding; tính đúng `in_features`), lưu `test_acc7`.

In [ ]:
X7, y7 = make_shapes(3000, seed=10)
X7_te, y7_te = make_shapes(1000, seed=11)
torch.manual_seed(0)
### START CODE HERE ### (≈ 12 dòng)
Xt7, Xe7 = torch.tensor(X7, dtype=torch.float32), torch.tensor(X7_te, dtype=torch.float32)        # 🐛 (1)
model7 = nn.Sequential(nn.Conv2d(1, 16, 3), nn.ReLU(), nn.MaxPool2d(2),                             # 28 → 26 → 13
                       nn.Conv2d(16, 32, 3), nn.ReLU(), nn.MaxPool2d(2),                            # 13 → 11 → 5
                       nn.Flatten(), nn.Dropout(0.3), nn.Linear(32 * 7 * 7, 5))                     # 🐛 (2)
### END CODE HERE ###
yt7, ye7 = torch.tensor(y7), torch.tensor(y7_te)
model7.to(device)
opt7 = torch.optim.Adam(model7.parameters(), lr=1e-3)
g7 = torch.Generator().manual_seed(0)
for epoch in range(15):
    model7.train()
    perm = torch.randperm(len(Xt7), generator=g7)
    for s in range(0, len(perm), 64):
        idx = perm[s:s + 64]
        opt7.zero_grad()
        nn.functional.cross_entropy(model7(Xt7[idx].to(device)), yt7[idx].to(device)).backward()
        opt7.step()
### START CODE HERE ### (≈ 3 dòng)
with torch.no_grad():                                                                               # 🐛 (3)
    test_acc7 = (model7(Xt7.to(device)).argmax(1).cpu() == yt7).float().mean().item()
### END CODE HERE ###
print(f"accuracy test = {test_acc7:.3f}")

In [ ]:
check_greater(test_acc7, 0.75, "accuracy test > 0.75")
check_equal(model7[-1].in_features, 32 * 5 * 5, "in_features đúng với 28 → 26 → 13 → 11 → 5")

## L5 — Phân tích

**Bài 8.** **Hiệu quả dữ liệu.** Train MLP (784 → 256 → 5) và CNN có global average pooling (3 lớp conv 16/32/64, như bài 02) với số mẫu train $n \in \{200, 600, 2000\}$ (`make_shapes(n, seed=20)`). Để so sánh công bằng, **cố định 1 000 bước cập nhật** cho mọi $n$ (mỗi bước lấy ngẫu nhiên 64 mẫu từ tập train — nếu cố định số epoch, $n$ nhỏ sẽ có rất ít bước cập nhật và ta lẫn lộn "ít dữ liệu" với "train ít"). Adam lr 1e-3; đánh giá trên cùng 1 000 mẫu test (`seed=21`). Lưu `acc8[(model, n)]`. Nhận xét.

In [ ]:
X8_te, y8_te = make_shapes(1000, seed=21)
Xe8, ye8 = to_tensor(X8_te), torch.tensor(y8_te)


def make_model8(kind):
    if kind == "MLP":
        return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 5))
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
                         nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 5))


acc8 = {}
### START CODE HERE ### (≈ 16 dòng)
# với n trong [200, 600, 2000] và kind trong ["MLP", "CNN"]: dữ liệu make_shapes(n, seed=20);
# torch.manual_seed(0); mô hình; Adam lr 1e-3; 1 000 bước, mỗi bước idx = torch.randint(0, n, (64,), generator=g) (g seed 0)
# acc8[(kind, n)] = accuracy trên Xe8 (nhớ model.eval())
pass
### END CODE HERE ###
for n in [200, 600, 2000]:
    print(f"n = {n:4d}: MLP = {acc8[('MLP', n)]:.3f} | CNN = {acc8[('CNN', n)]:.3f}")

In [ ]:
check_greater(acc8[("CNN", 200)], acc8[("MLP", 2000)] + 0.15, "CNN với 200 ảnh vượt MLP với 2000 ảnh")
check_true(all(acc8[("CNN", n)] > acc8[("MLP", n)] + 0.2 for n in [200, 600, 2000]), "CNN vượt MLP ở mọi n")
check_greater(acc8[("MLP", 2000)], acc8[("MLP", 200)] + 0.05, "MLP cải thiện chậm theo dữ liệu")

## L6 — Thực tế

**Bài 9.** Xây CNN cho bộ shapes (train: `make_shapes(4000, seed=30)`, test: `make_shapes(1000, seed=31)`) đạt accuracy test **≥ 0.95** với **≤ 100 000 tham số**. Gợi ý: BatchNorm, nhiều epoch hơn, augmentation dịch/xoay, global pooling. Lưu `model9`, `test_acc9`.

In [ ]:
X9, y9 = make_shapes(4000, seed=30)
X9_te, y9_te = make_shapes(1000, seed=31)
Xt9, Xe9, yt9, ye9 = to_tensor(X9), to_tensor(X9_te), torch.tensor(y9), torch.tensor(y9_te)
### START CODE HERE ### (≈ 25 dòng)
model9 = None
test_acc9 = None
### END CODE HERE ###
n_params9 = sum(p.numel() for p in model9.parameters())
print(f"{n_params9:,} tham số | accuracy test = {test_acc9:.3f}")

In [ ]:
check_less(n_params9, 100_001, "≤ 100 000 tham số")
check_greater(test_acc9, 0.95, "accuracy test ≥ 0.95")

## L7 — Nâng cao

**Bài 10.** **Backward của tích chập.** Với $Y = \text{conv}(X, W) + b$ (stride 1, padding $p$), cho $dY$:
- $db_o = \sum_{n, i, j} dY[n, o, i, j]$;
- $dW[o, c, a, b'] = \sum_{n, i, j} dY[n, o, i, j]\,X_p[n, c, i + a, j + b']$ (tương quan giữa đầu vào và $dY$);
- $dX_p[n, c, i + a, j + b'] \mathrel{+}= \sum_o dY[n, o, i, j]\,W[o, c, a, b']$ (rồi bỏ phần padding) — tương đương **tích chập đầy đủ** của $dY$ với kernel **lật**.

Viết `conv2d_backward(dY, x, w, padding)` → `(dx, dw, db)` (stride 1), vector hoá theo $k^2$ vị trí kernel. Kiểm tra với autograd.

In [ ]:
def conv2d_backward(dY, x, w, padding=0):
    ### START CODE HERE ### (≈ 12 dòng)
    return None, None, None
    ### END CODE HERE ###

In [ ]:
x10 = rng.normal(size=(3, 2, 9, 8))
w10, b10 = rng.normal(size=(4, 2, 3, 3)), rng.normal(size=4)
for padding in [0, 1]:
    tx, tw, tb = (torch.tensor(a, requires_grad=True) for a in (x10, w10, b10))
    ty = F.conv2d(tx, tw, tb, padding=padding)
    dY = rng.normal(size=ty.shape)
    ty.backward(torch.tensor(dY))
    dx, dw, db = conv2d_backward(dY, x10, w10, padding)
    check_close(dx, tx.grad.numpy(), tol=1e-10, message=f"dX (padding {padding})")
    check_close(dw, tw.grad.numpy(), tol=1e-10, message=f"dW (padding {padding})")
    check_close(db, tb.grad.numpy(), tol=1e-10, message=f"db (padding {padding})")